# Calculator Model Training and SHAP/FFA Workflow

**Purpose:** Train calculator models and run SHAP + Formal Feature Attribution (FFA) analysis  
**Updated:** January 27, 2026  
**Hardware:** Optimized for EC2 instances  
**Model Strategy:** **One model per cohort** (CHD, Myocardio, Combined) with **multiple variants** per cohort. The notebook can train and compare **ALL** variants: **base**, **enhanced**, **top 15**, **Wisotzkey**, and **FULL** (calculator + R replication vars). The **deployed** model per cohort is the best of top vs Wisotzkey by C-index (then AU-PRC).

## Overview

This notebook provides an interactive workflow for:

1. **Training Calculator Models** - Train **all selected variants per cohort** (default: base, enhanced, top 15, Wisotzkey, FULL) for each of CHD, Myocardio, Combined. Toggle `RUN_*` flags in the training cell to run a subset. The **Compare ALL Models** cell then loads every `mc_cv_model_metrics.csv` and shows C-index (and metrics) by cohort × variant.
2. **SHAP + FFA Analysis** - Generate causal factors and dashboard data per cohort (`--model-variant top`)
3. **Results Inspection** - View top causal factors, feature importance, aggregated importance (across CatBoost, XGBoost, XGBoost RF), model performance (C-index, Recall, AUC, AU-PRC), and compare top vs Wisotzkey via `compare_top_vs_wisotzkey.py`
4. **Dashboard Deployment** - Run compare_top_vs_wisotzkey.py --set-deployed, then prepare_lambda_dir_phts.py (copies both variants + deployed-variant file). Dashboard uses the **chosen model per cohort** (top or Wisotzkey) and shows best model chosen + all metrics (Risk Calculator, Causal Analysis, Aggregated Feature Importance, Documentation tabs)

## Model Architecture

- **One model per cohort, two variants each**: CHD, Myocardio, and Combined each have **two** model variants: (1) **top 15 features** (from SHAP/FFA importance/causality) and (2) **Wisotzkey** (Wisotzkey et al. variable set). The notebook training cell produces both variants for all cohorts by default.
  - **Top models**: `CHD_top/`, `Myocardio_top/`, `Combined_top/` — produced by the notebook training cell (or `python train_python_models.py --cohort <Cohort> --top_features_only`)
  - **Wisotzkey variants**: `CHD_wisotzkey/`, `Myocardio_wisotzkey/`, `Combined_wisotzkey/` — produced by the notebook training cell (or `python train_python_models.py --cohort <Cohort> --wisotzkey_vars_only`). See `wisotzkey_data.py`, `calculator/wisotzkey/README_WISOTZKEY.md`. Same MC-CV and metrics (C-index, Recall, AUC, AU-PRC).
  - **Deployed model per cohort**: The **best of top vs Wisotzkey** by C-index (then AU-PRC). Run `compare_top_vs_wisotzkey.py --set-deployed` to write `{cohort}_deployed_variant.txt`; Lambda and dashboard then use that variant. **base**, **enhanced**, and **FULL** variants are trained and compared in the notebook for C-index analysis (e.g. to assess cohort segregation vs feature set); they are not used for deployment.
- **Feature list (top models)**: In `top_causal_features.py` (e.g. sec_dx, donor_age, txalt, lbun_r, txbun_r, egfr_change, chd_sv, donor_size_ratio, hxsurg, lsbaosat, bmi_txpl, lstp_r, egfr_tx, donor_weight_ratio, txsa_r). Each cohort has its own **sec_dx** options in dashboard data.
- **Feature Engineering**: Same derived variables as full calculator; top models use only the top 15 inputs per cohort; Wisotzkey models use the fixed Wisotzkey feature set per cohort.

## Model Training Strategy

**Per-cohort variants:**
1. **Top models** (`CHD_top`, `Myocardio_top`, `Combined_top`): Top 15 causal/importance features only (~15 features per cohort).
2. **Wisotzkey variants** (`CHD_wisotzkey`, `Myocardio_wisotzkey`, `Combined_wisotzkey`): Wisotzkey et al. variable set only (`--wisotzkey_vars_only`). Same training pipeline and metrics; required part of the workflow.
3. **Best model chosen**: Per cohort, the **deployed** model is whichever variant (top or Wisotzkey) has higher C-index, then AU-PRC. Run `compare_top_vs_wisotzkey.py --set-deployed` after training both; Lambda uses the resulting `{cohort}_deployed_variant.txt`. At inference, **Top** uses calculator-derived features; **Wisotzkey** uses the Wisotzkey feature set (Lambda builds the appropriate inputs per variant).

**Three model types trained (per cohort, per variant):**
1. **CatBoost** - Gradient boosting with categorical feature support (Cox regression)
2. **XGBoost** - Extreme gradient boosting (Cox regression)
3. **XGBoost Random Forest** - XGBoost in Random Forest mode (Cox regression)

**Model selection and metrics:**
- For each variant (top or Wisotzkey), all three types are trained on that variant's feature set; best model is selected per variant.
- **Standard metrics** (C-index, Recall, AUC, AU-PRC) are computed and saved; best model by **C-index** (then **AU-PRC**) is selected and saved to `{cohort}_top/` or `{cohort}_wisotzkey/` respectively.
- **Aggregated feature importance** (mean ± std across the three model types over MC-CV splits) is written to `mc_cv_aggregated_feature_importance.csv` and shown in the dashboard’s Aggregated Feature Importance tab.
- **Training default**: New runs retrain (force=True). Use `--no-force` to skip when outputs already exist.

**Dashboard deployment:**
- **Risk Calculator tab**: Uses the **deployed variant** for the selected cohort (top or Wisotzkey).
- **Causal Analysis tab**: Uses that cohort’s dashboard data and sec_dx options.
- **Aggregated Feature Importance tab**: Shows mean importance across CatBoost, XGBoost, XGBoost RF per cohort.
- **Documentation tab**: Shows **best model chosen** (algorithm + Top/Wisotzkey) and all four metrics per cohort from the API.

## Causal Analysis Strategy (SHAP + FFA)

**Important:** The causal analysis workflow uses a specific combination of models and applies rules to the **test set** for final causal analysis.

### Workflow Overview

```mermaid
graph TD
    A[Training Data] --> B[Temporal Split<br/>80/20]
    B --> C[Train Set<br/>txpl_year ≤ cutoff]
    B --> D[Test Set<br/>txpl_year > cutoff]
    
    C --> E[Train Models<br/>CatBoost, XGBoost, XGBoost RF]
    E --> F[Select Best Model<br/>by C-index, then AU-PRC]
    F --> G[Final Model<br/>Trained on Train Set]
    
    G --> H[Extract Rules<br/>from XGBoost JSON]
    D --> I[Compute SHAP Values<br/>on Test Set Only]
    
    H --> J[Apply Rules to Test Set<br/>Count Rule Firings]
    I --> K[Combine SHAP Values<br/>XGBoost + CatBoost if needed]
    
    J --> L[Calculate Rule Frequencies<br/>from Test Set]
    K --> M[SHAP Importance<br/>per Feature]
    
    L --> N[Causal Responsibility<br/>rule_freq × SHAP_importance]
    M --> N
    
    N --> O[Top K Causal Factors<br/>for Dashboard]
    
    style D fill:#e1f5ff
    style I fill:#e1f5ff
    style J fill:#e1f5ff
    style L fill:#e1f5ff
    style O fill:#c8e6c9
```

### Key Principles

**1. Test Set Application (Critical)**
- ✅ **Rules are extracted from the trained model** (trained on training set)
- ✅ **Rules are applied to the test set** (unseen data) for final causal analysis
- ✅ **SHAP values are computed on the test set only** (not training set)
- ✅ **Rule frequencies are counted from test set rule firings** (not from rule definitions)
- ✅ **Temporal split cutoff matches training** (dynamic 80/20 split, falls back to 2021)

**Why Test Set?**
- Ensures causal factors reflect model behavior on **unseen data**
- Prevents overfitting to training patterns
- Provides realistic causal responsibility scores
- Matches model evaluation methodology

### SHAP Values (Feature Importance)
- **Best XGBoost Model**: SHAP values are **always** computed from the best XGBoost model
- **Best CatBoost Model**: SHAP values are computed from the best CatBoost model **only if CatBoost is the best model**
- **Combination**: If CatBoost is best, SHAP values are combined with **auto-determined weights** based on C-index values
- **If XGBoost is best**: Only XGBoost SHAP values are used
- **Data Source**: SHAP values computed on **test set only** (`txpl_year > cutoff_year`)

### FFA Analysis (Rule Extraction)
- **Best XGBoost JSON Model**: Rules are **always** extracted from the best XGBoost JSON model
  - This is because XGBoost JSON structure is easier to parse for rule extraction
  - CatBoost JSON is not used (harder to parse due to categorical hashing)
- **Rule Source**: Rules extracted from model trained on **training set**
- **Rule Application**: Rules are **applied to test set instances** to count actual rule firings
- **Rule Filtering**: Rules are filtered using SHAP importance values (from step above)
- **Causal Responsibility**: Calculated as `(rule_frequency_from_test_set / total_rule_firings) × SHAP_importance`
  - `rule_frequency_from_test_set`: Count of how many times a rule fires on test set instances
  - `SHAP_importance`: Feature importance from SHAP values (computed on test set)

### Summary
**For causal analysis:**
1. ✅ SHAP values from **best XGBoost model** (always, computed on **test set**)
2. ✅ SHAP values from **best CatBoost model** (only if CatBoost is best, computed on **test set**)
3. ✅ Rules extracted from **best XGBoost JSON model** (trained on **training set**)
4. ✅ Rules applied to **test set** to count actual rule firings
5. ✅ FFA analysis combines **test set rule frequencies** + **test set SHAP values** to calculate causal responsibility

## Workflow Steps

- **Step 1:** Run the **Training Calculator Models** cell. It trains **all selected cohorts** (default: CHD, Myocardio, Combined) and **all selected variants** per cohort: base, enhanced, top 15, Wisotzkey, FULL. Toggle `RUN_BASE`, `RUN_ENHANCED`, `RUN_TOP`, `RUN_WISOTZKEY`, `RUN_FULL` to run a subset. Outputs: `{cohort}_base/`, `{cohort}_enhanced/`, `{cohort}_top/`, `{cohort}_wisotzkey/`, `{cohort}_FULL/`.
- **Step 1b:** Run the **Compare ALL Models** cell to load every `mc_cv_model_metrics.csv` and view C-index (and metrics) by cohort × variant.
- **Step 2:** Run SHAP/FFA analysis per cohort: `run_shap_ffa_workflow.py --cohort <Cohort> --model-variant top` → dashboard data and causal factors per cohort
- **Step 3:** Inspect results: MC-CV metrics, aggregated feature importance, and `compare_top_vs_wisotzkey.py` to compare top vs Wisotzkey; export dashboard data for each `{cohort}_top`
- **Step 4:** Deploy: run `compare_top_vs_wisotzkey.py --set-deployed`, then `prepare_lambda_dir_phts.py` (copies `*_top` and `*_wisotzkey` models and deployed-variant file), then build/update Lambda and upload dashboard HTML

The notebook **trains all three cohorts with all five variants by default** (`COHORT = None`, all `RUN_* = True`). Set `COHORT` to a single cohort or any `RUN_* = False` to reduce scope.

## Expected Runtime

- **Per variant per cohort:** ~15–30 minutes
- **All five variants × three cohorts:** ~225–450+ minutes on EC2 (set some `RUN_* = False` or single `COHORT` to reduce)
- **SHAP/FFA (per cohort):** ~10–20 minutes


## 1. Input Features Overview

### Required Input Variables for Risk Calculator

The model uses the following input features, with automatic feature engineering for derived variables:

#### Primary Diagnosis & History
- **Primary Diagnosis** (`primary_etiology`) - Congenital Heart Disease, Cardiomyopathy, Myocarditis, Other
- **Previous Cardiac Surgery** (`hxsurg`) - History of surgery (Yes/No)
- **Laterality Disorder** (`chd_lat`) - Composite variable (Yes/No)
  - Derived from: `chd_dex`, `chd_si`, `chd_heter`, `chd_iivc`, `chd_bivc`, `chd_lsvc`, `chd_raa`, `chd_avd`

#### Cardiac Support Devices (Combined Variables)
- **ECMO** (`ecmo_combined`) - ECMO at transplant OR listing
  - Derived from: `txecmo` OR `slecmo`
- **VAD** (`vad_combined`) - VAD at transplant OR listing
  - Derived from: `txvad` OR `slvad`
- **Mechanical Ventilation** (`vent_combined`) - Ventilation at transplant OR listing
  - Derived from: `txvent` OR `slvent` OR `ltxtrach` OR `hxtrach`

#### Demographics & Age
- **Age at Transplant** (`age_txpl`) - Years (priority over `age_listing`)
- **Age at Listing** (`age_listing`) - Years (fallback)

#### Renal Function
- **Dialysis History** (`hxdysdia` / `hxdysdia_bin`) - History of dialysis (ever)
- **eGFR at Transplant** (`egfr_tx`) - Calculated from height and creatinine
  - Formula: `egfr_tx = 0.413 × height_txpl / txcreat_r`
- **eGFR at Listing** (`egfr_listing`) - Calculated from height and creatinine

#### Liver Function
- **ALT at Transplant** (`txalt`) - U/L (priority over `lsalt`)
- **AST at Transplant** (`txast`) - U/L (priority over `lsast`)
- **Direct Bilirubin at Transplant** (`txbili_d_r`) - mg/dL (priority over `lsbili_d_r`)
- **Total Bilirubin at Transplant** (`txbili_t_r`) - mg/dL (priority over `lsbili_t_r`)

#### Nutrition
- **Serum Albumin at Transplant** (`txsa_r`) - g/dL (priority over `lssab_r`)
- **Total Protein at Transplant** (`txtp_r`) - g/dL (priority over `lstp_r`)

#### Immunology
- **cPRA at Transplant** (`txfcpra`) - Flow cytometry PRA % (priority over `lsfcpra`)
- **cPRA at Listing** (`lsfcpra`) - Flow cytometry PRA % (fallback)

#### Donor Characteristics
- **Donor Ischemic Time** (`donisch`) - Minutes (default: < 240 minutes if not provided)
- **Donor/Recipient Weight Ratio** (`donor_weight_ratio`) - Percentage
  - Formula: `(weight_donor / weight_txpl) × 100`
  - Model assumption: 70-200%
- **Donor/Recipient Size Ratio** (`donor_size_ratio`) - Percentage
  - Formula: `(height_donor / height_txpl) × 100`
  - Model assumption: 70-200%

### Additional Features

The model also includes:
- All CHD subtype variables (40+ subtypes, e.g., `chd_hlh`, `chd_lsvc`, `chd_si`, etc.)
- Additional lab values and clinical history variables
- Derived categorical variables (eGFR categories, high/low indicators)
- Donor characteristics and transplant details

### Feature Engineering

The following variables are automatically created during training and inference:
1. `ecmo_combined` - ECMO combined
2. `vad_combined` - VAD combined
3. `vent_combined` - Ventilation combined
4. `donor_weight_ratio` - Donor/recipient weight ratio
5. `donor_size_ratio` - Donor/recipient height ratio
6. `chd_lat` - Laterality disorder composite
7. `egfr_tx` - eGFR at transplant (if not provided, calculated from height/creatinine)
8. `egfr_listing` - eGFR at listing
9. `egfr_tx_cat` - eGFR category (severe/moderate/mild/normal)
10. `egfr_listing_cat` - eGFR category at listing
11. Additional derived variables (BMI, high/low indicators, etc.)

---

## 2. Setup and Configuration

Load required packages and configure paths.

In [1]:
import sys
from pathlib import Path
import logging
import warnings
warnings.filterwarnings('ignore')

# Add project paths
PROJECT_ROOT = Path().resolve().parent.parent.parent
CALCULATOR_DIR = Path().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(CALCULATOR_DIR))

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("=" * 80)
print("PHTS Calculator Workflow")
print("=" * 80)
print(f"Project root: {PROJECT_ROOT}")
print(f"Calculator directory: {CALCULATOR_DIR}")
print("=" * 80)

PHTS Calculator Workflow
Project root: /home/pgx3874/phts
Calculator directory: /home/pgx3874/phts/graft-loss/cohort_analysis/calculator


In [2]:
# Check Docker service
import subprocess
import platform

def check_docker():
    """Simple Docker check - verify if Docker is accessible."""
    print("\n" + "=" * 80)
    print("Docker Check")
    print("=" * 80)
    
    try:
        result = subprocess.run(
            ["docker", "ps"],
            capture_output=True,
            text=True,
            timeout=5
        )
        
        if result.returncode == 0:
            print("✓ Docker is running")
            logger.info("Docker is accessible")
            return True
        else:
            print("⚠ Docker is not accessible")
            if "permission denied" in result.stderr.lower():
                print("  Permission issue - you may need to add user to docker group:")
                print("    Linux: sudo usermod -aG docker $USER && newgrp docker")
            else:
                print(f"  Error: {result.stderr.strip()}")
            return False
            
    except FileNotFoundError:
        print("✗ Docker not found - please install Docker")
        system = platform.system()
        if system == "Windows":
            print("  Install Docker Desktop from: https://www.docker.com/products/docker-desktop")
        else:
            print("  Linux: https://docs.docker.com/engine/install/")
            print("  macOS: Install Docker Desktop")
        return False
    except Exception as e:
        print(f"⚠ Error checking Docker: {e}")
        return False
    
    print("=" * 80)

# Run check
docker_ok = check_docker()

2026-02-16 02:11:48,628 - __main__ - INFO - Docker is accessible



Docker Check
✓ Docker is running



Docker Check
✓ Docker is running


In [3]:
# Timing helper for workflow steps (aligned with mermaid chart workflow)
import time
from contextlib import contextmanager

@contextmanager
def step_timer(step_name, sub_steps=None):
    """
    Context manager to time workflow steps with logging.
    
    Aligns with mermaid chart workflow:
    - Training: Temporal Split → Train Models → Select Best Model → Final Model
    - SHAP/FFA: Extract Rules → Compute SHAP → Apply Rules → Calculate Frequencies → Causal Responsibility
    
    Args:
        step_name: Main step name (e.g., "Step 1: Train Baseline Model")
        sub_steps: Optional list of sub-steps that align with mermaid chart nodes
    """
    start_time = time.time()
    start_str = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(start_time))
    print(f"\n{'=' * 80}")
    print(f"⏱️  START: {step_name}")
    print(f"   Started at: {start_str}")
    if sub_steps:
        print(f"   Sub-steps (per mermaid chart):")
        for i, sub_step in enumerate(sub_steps, 1):
            print(f"     {i}. {sub_step}")
    print(f"{'=' * 80}")
    logger.info(f"START: {step_name} at {start_str}")
    if sub_steps:
        logger.info(f"Sub-steps: {', '.join(sub_steps)}")
    
    try:
        yield
    finally:
        end_time = time.time()
        end_str = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(end_time))
        duration = end_time - start_time
        duration_min = duration / 60
        duration_sec = duration % 60
        
        print(f"\n{'=' * 80}")
        print(f"✅ COMPLETE: {step_name}")
        print(f"   Started: {start_str}")
        print(f"   Finished: {end_str}")
        print(f"   Duration: {duration_min:.1f} minutes ({duration:.0f} seconds)")
        print(f"{'=' * 80}")
        logger.info(f"COMPLETE: {step_name} - Duration: {duration_min:.1f} minutes ({duration:.0f} seconds)")

print("✓ Timing helper loaded (aligned with mermaid chart workflow)")

✓ Timing helper loaded (aligned with mermaid chart workflow)


In [4]:
# Configuration
DEBUG_MODE = False  # Set to True for quick testing (fewer splits)

# Model Strategy: One model per cohort (CHD, Myocardio, Combined); two variants per cohort (top, Wisotzkey)
COHORTS = ["CHD", "Myocardio", "Combined"]  # All cohorts; default is to run all three
COHORT = None  # None = run all cohorts (CHD, Myocardio, Combined); or set to "CHD", "Myocardio", or "Combined" for single cohort

# SHAP/FFA configuration
TOP_K = 15  # Number of top causal factors to extract
# Note: Weights are automatically determined from best model C-index values
# Set to None to use auto-determination, or override manually if needed
WEIGHT_CATBOOST = None  # Auto-determined from best model (None = auto)
WEIGHT_XGBOOST = None   # Auto-determined from best model (None = auto)

print(f"\nConfiguration:")
print(f"  DEBUG_MODE: {DEBUG_MODE}")
print(f"  Cohorts: {COHORTS}")
print(f"  Run: {'All cohorts (CHD, Myocardio, Combined)' if COHORT is None else COHORT}")
print(f"  Top K factors: {TOP_K}")
print(f"  SHAP Weights: Auto-determined from best model C-index values")
print(f"    (CatBoost weight: {'Auto' if WEIGHT_CATBOOST is None else WEIGHT_CATBOOST})")
print(f"    (XGBoost weight: {'Auto' if WEIGHT_XGBOOST is None else WEIGHT_XGBOOST})")
print(f"\nNote: The model includes primary_etiology to distinguish between:")
print(f"  - Congenital Heart Disease")
print(f"  - Cardiomyopathy")
print(f"  - Myocarditis")
print(f"  - Other")


Configuration:
  DEBUG_MODE: False
  Cohorts: ['CHD', 'Myocardio', 'Combined']
  Run: All cohorts (CHD, Myocardio, Combined)
  Top K factors: 15
  SHAP Weights: Auto-determined from best model C-index values
    (CatBoost weight: Auto)
    (XGBoost weight: Auto)

Note: The model includes primary_etiology to distinguish between:
  - Congenital Heart Disease
  - Cardiomyopathy
  - Myocarditis
  - Other


In [5]:
# Check dependencies
print("\nChecking dependencies...")

try:
    import numpy as np
    import pandas as pd
    from catboost import CatBoostRegressor
    import xgboost as xgb
    import shap
    print("✓ All required packages are installed")
    print(f"  NumPy: {np.__version__}")
    print(f"  Pandas: {pd.__version__}")
    print(f"  XGBoost: {xgb.__version__}")
    print(f"  SHAP: {shap.__version__}")
except ImportError as e:
    print(f"✗ Missing dependency: {e}")
    print("  Please install: pip install numpy pandas catboost xgboost shap")


Checking dependencies...
✓ All required packages are installed
  NumPy: 1.26.4
  Pandas: 2.2.3
  XGBoost: 3.1.2
  SHAP: 0.47.2


In [6]:
# Check data availability
print("\nChecking data availability...")

data_file = PROJECT_ROOT / "graft-loss" / "data" / "phts_txpl_ml.sas7bdat"
if data_file.exists():
    size_mb = data_file.stat().st_size / (1024 * 1024)
    print(f"✓ Data file found: {data_file}")
    print(f"  Size: {size_mb:.2f} MB")
else:
    print(f"⚠ Data file not found: {data_file}")
    print("  You may need to download the data file first")

# Check calculator directory structure
outputs_dir = CALCULATOR_DIR / "outputs"
if outputs_dir.exists():
    print(f"✓ Outputs directory exists: {outputs_dir}")
else:
    print(f"✓ Creating outputs directory: {outputs_dir}")
    outputs_dir.mkdir(parents=True, exist_ok=True)


Checking data availability...
✓ Data file found: /home/pgx3874/phts/graft-loss/data/phts_txpl_ml.sas7bdat
  Size: 25.50 MB
✓ Outputs directory exists: /home/pgx3874/phts/graft-loss/cohort_analysis/calculator/outputs


## 3. Train Calculator Models

Train **all three model types** (CatBoost, XGBoost, and XGBoost RF) for **all selected cohorts** (default: CHD, Myocardio, Combined) and **both variants** (top 15 features + Wisotzkey) per cohort. With `COHORT = None`, the cell below trains six model sets: `CHD_top`, `CHD_wisotzkey`, `Myocardio_top`, `Myocardio_wisotzkey`, `Combined_top`, `Combined_wisotzkey`. Set `COHORT` to a single cohort (e.g. `"Combined"`) to train only that cohort (still both variants).

**Training Process (per cohort, per variant):**
1. **Data Split**: Temporal 80/20 split (train on earlier years, test on later years)
2. **Model Training**: All three models are trained on that variant's feature set:
   - CatBoost (Cox regression)
   - XGBoost (Cox regression)
   - XGBoost Random Forest (Cox regression)
3. **Model Evaluation**: C-index (and Recall, AUC, AU-PRC) are calculated for each model on the test set
4. **Model Selection**: The model with the highest C-index (then AU-PRC) is selected and saved to `{cohort}_top/` or `{cohort}_wisotzkey/`

**Note:** After training, run `compare_top_vs_wisotzkey.py --set-deployed` to choose the deployed variant per cohort (best by C-index then AU-PRC) for the dashboard.

## 2b. Wisotzkey-vars cohort datasets (same SAS source)

The pipeline uses a **single SAS dataset** (`phts_txpl_ml.sas7bdat`) for all cohorts. This step builds the **Wisotzkey et al.** variable set (Wisotzkey et al., *Pediatric Transplantation* 2023) for each cohort from that same source:

- **CHD**: Congenital HD only  
- **Myocardio**: Cardiomyopathy + Myocarditis  
- **Combined**: All three etiologies  

Outputs are written to `outputs/wisotzkey/wisotzkey_CHD.csv`, `wisotzkey_Myocardio.csv`, and `wisotzkey_Combined.csv`. These can be used for R replication (e.g. `scripts/R/wisotzkey-vars.R`) or for training Wisotzkey-based models per cohort.

### 1. Top Model (Top 15 Causal Features)

In [ ]:
# Import training function
from train_python_models import train_models_for_cohort
import os
import multiprocessing

# Variants to train: base, enhanced, top, wisotzkey, FULL (set to False to skip a variant)
RUN_BASE = True
RUN_ENHANCED = True
RUN_TOP = True
RUN_WISOTZKEY = True
RUN_FULL = True

# Determine number of parallel jobs (use all available CPUs minus 1 for safety)
n_parallel_jobs = max(1, multiprocessing.cpu_count() - 1)

# Which cohorts to run: all when COHORT is None, else single cohort
cohorts_to_run = COHORTS if COHORT is None else [COHORT]
variants = []
if RUN_BASE: variants.append("base")
if RUN_ENHANCED: variants.append("enhanced")
if RUN_TOP: variants.append("top")
if RUN_WISOTZKEY: variants.append("wisotzkey")
if RUN_FULL: variants.append("FULL")

print(f"\n{'=' * 80}")
print("Training Calculator Models: ALL variants (per cohort)")
print(f"{'=' * 80}")
print(f"\nConfiguration:")
print(f"  Cohorts: {cohorts_to_run}")
print(f"  Variants per cohort: {variants}")
print(f"  Parallel Jobs: {n_parallel_jobs} (using {multiprocessing.cpu_count()} CPUs)")
print(f"  MC-CV Splits: 25")
print(f"  Training Proportion: 80%")
print(f"\nTraining Process:")
print(f"  1. Monte Carlo Cross-Validation (25 splits)")
print(f"  2. Model Training: All three model types (CatBoost, XGBoost, XGBoost RF)")
print(f"  3. Model Selection: Best model by C-index, then AU-PRC")
print(f"  4. Final Model: Best model trained on full temporal split")
print("-" * 80)

try:
    import time
    from top_causal_features import get_top_causal_features
    start_time = time.time()
    for c in cohorts_to_run:
        print(f"\n--- Cohort: {c} ---")
        if RUN_BASE:
            print(f"  Training {c}_base (calculator base features)...")
            train_models_for_cohort(cohort=c, n_mc_splits=25, train_prop=0.8, n_jobs=n_parallel_jobs, include_recommended_features=False)
            print(f"  ✓ {c}_base done -> outputs/models/{c}_base/")
        if RUN_ENHANCED:
            print(f"  Training {c}_enhanced (base + recommended features)...")
            train_models_for_cohort(cohort=c, n_mc_splits=25, train_prop=0.8, n_jobs=n_parallel_jobs, include_recommended_features=True)
            print(f"  ✓ {c}_enhanced done -> outputs/models/{c}_enhanced/")
        if RUN_TOP:
            print(f"  Training {c}_top (top 15 features)...")
            train_models_for_cohort(cohort=c, n_mc_splits=25, train_prop=0.8, n_jobs=n_parallel_jobs, include_recommended_features=False, top_feature_names=get_top_causal_features())
            print(f"  ✓ {c}_top done -> outputs/models/{c}_top/")
        if RUN_WISOTZKEY:
            print(f"  Training {c}_wisotzkey (Wisotzkey vars)...")
            train_models_for_cohort(cohort=c, n_mc_splits=25, train_prop=0.8, n_jobs=n_parallel_jobs, include_recommended_features=False, use_wisotzkey_vars_only=True)
            print(f"  ✓ {c}_wisotzkey done -> outputs/models/{c}_wisotzkey/")
        if RUN_FULL:
            print(f"  Training {c}_FULL (calculator + R replication vars)...")
            train_models_for_cohort(cohort=c, n_mc_splits=25, train_prop=0.8, n_jobs=n_parallel_jobs, include_recommended_features=False, use_full_feature_set=True)
            print(f"  ✓ {c}_FULL done -> outputs/models/{c}_FULL/")
    elapsed_time = time.time() - start_time
    print(f"\n✓ All cohort and variant training complete!")
    print(f"  Total time: {elapsed_time/60:.1f} minutes ({elapsed_time:.0f} seconds)")
    print(f"  Next: Run the 'Compare ALL Models' cell below, then compare_top_vs_wisotzkey.py --set-deployed, SHAP/FFA and deployment.")
except Exception as e:
    print(f"\n✗ Error during training: {e}")
    import traceback
    traceback.print_exc()

print(f"\n{'=' * 80}")
print("ALL Models Training Complete!")
print(f"{'=' * 80}")

2026-02-16 02:11:56,931 - botocore.credentials - INFO - Found credentials from IAM Role: EC2_Spot
2026-02-16 02:11:57,088 - train_python_models - INFO - 
2026-02-16 02:11:57,089 - train_python_models - INFO - Training models for cohort: CHD
2026-02-16 02:11:57,089 - train_python_models - INFO - ================================================================================
2026-02-16 02:11:57,089 - train_python_models - INFO - MC-CV Configuration:
2026-02-16 02:11:57,090 - train_python_models - INFO -   - Number of splits: 25
2026-02-16 02:11:57,090 - train_python_models - INFO -   - Training proportion: 80.0%
2026-02-16 02:11:57,090 - train_python_models - INFO -   - Parallel jobs: 31
2026-02-16 02:11:57,090 - train_python_models - INFO -   - Time horizon for AUC/AU-PRC/Recall: 365.25 days
2026-02-16 02:11:57,090 - train_python_models - INFO - 
2026-02-16 02:11:57,167 - run_shap_ffa_workflow - INFO - FFA modules loaded using direct file import
2026-02-16 02:11:57,168 - train_python_m


Training Calculator Models: Top 15 + Wisotzkey (per cohort)

Configuration:
  Cohorts: ['CHD', 'Myocardio', 'Combined']
  Variants per cohort: Top 15 + Wisotzkey
  Parallel Jobs: 31 (using 32 CPUs)
  MC-CV Splits: 25
  Training Proportion: 80%

Training Process:
  1. Monte Carlo Cross-Validation (25 splits)
  2. Model Training: All three model types (CatBoost, XGBoost, XGBoost RF)
  3. Model Selection: Best model by C-index, then AU-PRC
  4. Final Model: Best model trained on full temporal split
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------

--- Cohort: CHD ---
  Training CHD_top (top 15 features)...


2026-02-16 02:11:57,367 - train_python_models - INFO - Loaded 2845 rows
2026-02-16 02:11:57,373 - run_shap_ffa_workflow - INFO - Calculated egfr_tx using Schwartz formula
2026-02-16 02:11:57,375 - run_shap_ffa_workflow - INFO - Calculated egfr_listing using Schwartz formula
2026-02-16 02:11:57,376 - run_shap_ffa_workflow - INFO - Calculated bmi_txpl
2026-02-16 02:11:57,378 - run_shap_ffa_workflow - INFO - Created egfr_tx_cat categories
2026-02-16 02:11:57,380 - run_shap_ffa_workflow - INFO - Created egfr_listing_cat categories
2026-02-16 02:11:57,380 - run_shap_ffa_workflow - INFO - Created txbili_t_r_high
2026-02-16 02:11:57,381 - run_shap_ffa_workflow - INFO - Created txbun_r_high from txbun_r
2026-02-16 02:11:57,381 - run_shap_ffa_workflow - INFO - Created txsa_r_low
2026-02-16 02:11:57,382 - run_shap_ffa_workflow - INFO - Created txalt_high
2026-02-16 02:11:57,382 - run_shap_ffa_workflow - INFO - Created ecmo_combined
2026-02-16 02:11:57,383 - run_shap_ffa_workflow - INFO - Created

[0]	train-cox-nloglik:7.30087	eval-cox-nloglik:5.93744
[0]	train-cox-nloglik:7.28543	eval-cox-nloglik:5.99242
[39]	train-cox-nloglik:6.94501	eval-cox-nloglik:5.94573
[78]	train-cox-nloglik:6.73850	eval-cox-nloglik:5.96005
[0]	train-cox-nloglik:7.32496	eval-cox-nloglik:5.82595
[0]	train-cox-nloglik:7.29798	eval-cox-nloglik:5.94619
[52]	train-cox-nloglik:6.90280	eval-cox-nloglik:5.89827
[36]	train-cox-nloglik:7.00507	eval-cox-nloglik:5.80622


2026-02-16 02:12:01,314 - train_python_models - INFO -   XGBoost C-index: 0.614352
2026-02-16 02:12:01,314 - train_python_models - INFO -   Saved XGBoost binary to /home/pgx3874/phts/graft-loss/cohort_analysis/calculator/outputs/models/CHD_top/mc_cv/split_13/xgboost_model.ubj
2026-02-16 02:12:01,319 - train_python_models - INFO -   Saved XGBoost JSON to /home/pgx3874/phts/graft-loss/cohort_analysis/calculator/outputs/models/CHD_top/mc_cv/split_13/final_model_json/CHD_final_model_xgboost.json (with 21 feature names)
2026-02-16 02:12:01,322 - scripts.py.feature_importance_model_utils - INFO - XGBoost importance: computing gain (Gini) for 21 features
2026-02-16 02:12:01,323 - scripts.py.feature_importance_model_utils - INFO - XGBoost importance: 14/21 features have gain_importance > 0 (env PGX_XGB_PERM_TOP_K=None is informational only)
2026-02-16 02:12:01,323 - scripts.py.feature_importance_model_utils - INFO - XGBoost importance: running permutation importance on full feature set (21 fea

...


Training Top-Features Calculator Model (Top 15 Causal/Importance Features Only)

Configuration:
  Cohort: Combined
  Feature Set: Top 15 only (from top_causal_features.py)
  Parallel Jobs: 31 (using 32 CPUs)
  MC-CV Splits: 25
  Training Proportion: 80%

Training Process:
  1. Monte Carlo Cross-Validation (25 splits)
  2. Model Training: All three model types (CatBoost, XGBoost, XGBoost RF)
  3. Model Selection: Best model by C-index, then AU-PRC
  4. Final Model: Best model trained on full temporal split
--------------------------------------------------------------------------------

Model Features (top 15): sec_dx, donor_age, txalt, lbun_r, txbun_r, egfr_change, chd_sv, donor_size_ratio, hxsurg, lsbaosat, bmi_txpl, lstp_r, egfr_tx, donor_weight_ratio, txsa_r
--------------------------------------------------------------------------------

Training top-features Combined model...
  Output directory: outputs/models/Combined_top/


2026-02-10 03:19:10,234 - run_shap_ffa_workflow - INFO - Created txbun_r_high from txbun_r
2026-02-10 03:19:10,234 - run_shap_ffa_workflow - INFO - Created txsa_r_low
2026-02-10 03:19:10,235 - run_shap_ffa_workflow - INFO - Created txalt_high
2026-02-10 03:19:10,235 - run_shap_ffa_workflow - INFO - Created ecmo_combined
2026-02-10 03:19:10,236 - run_shap_ffa_workflow - INFO - Created vad_combined
2026-02-10 03:19:10,238 - run_shap_ffa_workflow - INFO - Created vent_combined from ['txvent', 'slvent', 'ltxtrach', 'hxtrach']
2026-02-10 03:19:10,240 - run_shap_ffa_workflow - INFO - Created donor_weight_ratio
2026-02-10 03:19:10,241 - run_shap_ffa_workflow - INFO - Created donor_size_ratio
2026-02-10 03:19:10,243 - run_shap_ffa_workflow - INFO - Created chd_lat from ['chd_dex', 'chd_si', 'chd_heter', 'chd_iivc', 'chd_bivc', 'chd_lsvc', 'chd_raa', 'chd_avd']
2026-02-10 03:19:10,243 - run_shap_ffa_workflow - INFO - Created hxfonlvr_bin
2026-02-10 03:19:10,244 - run_shap_ffa_workflow - INFO - 


✓ Top-features model training complete!
  Total time: 0.0 minutes (0 seconds)
  Output saved to: outputs/models/Combined_top/

All three models have been trained; best model selected and saved.

Ready for SHAP/FFA analysis (--model-variant top) and dashboard deployment.

Top Model Training Complete!


2026-02-03 17:21:47,982 - run_shap_ffa_workflow - INFO - Loaded 5835 rows for cohort Combined
2026-02-03 17:21:47,982 - run_shap_ffa_workflow - INFO - Preparing calculator features (eGFR, BMI, dichotomous variables, etc.)...
2026-02-03 17:21:48,015 - run_shap_ffa_workflow - INFO - Calculated egfr_tx using Schwartz formula
2026-02-03 17:21:48,017 - run_shap_ffa_workflow - INFO - Calculated egfr_listing using Schwartz formula
2026-02-03 17:21:48,019 - run_shap_ffa_workflow - INFO - Calculated bmi_txpl
2026-02-03 17:21:48,020 - run_shap_ffa_workflow - INFO - Calculated age_txpl_months from age_txpl
2026-02-03 17:21:48,022 - run_shap_ffa_workflow - INFO - Created egfr_tx_cat categories
2026-02-03 17:21:48,024 - run_shap_ffa_workflow - INFO - Created egfr_listing_cat categories
2026-02-03 17:21:48,024 - run_shap_ffa_workflow - INFO - Created txbili_t_r_high
2026-02-03 17:21:48,025 - run_shap_ffa_workflow - INFO - Created txbun_r_high from txbun_r
2026-02-03 17:21:48,026 - run_shap_ffa_workf


✓ Baseline Combined model training complete!
  Total time: 0.0 minutes (0 seconds)
  Output saved to: outputs/models/Combined_base/

All three models have been trained:
  - CatBoost
  - XGBoost
  - XGBoost Random Forest

The best model (highest C-index) has been selected and saved.

The model is now ready for:
  - Risk prediction for all cohorts (CHD, Cardiomyopathy, Myocarditis)
  - SHAP/FFA analysis to extract causal factors

Baseline Model Training Complete!


### Compare ALL Models (C-index and metrics by cohort × variant)

After training, the cell below loads `mc_cv_model_metrics.csv` from every variant directory under `outputs/models/` and shows:
- **Best model per variant** (by C-index, then AU-PRC)
- **C-index comparison** across all cohort × variant combinations

Use this to see how C-index changes by feature set (base vs enhanced vs top vs wisotzkey vs FULL) and by cohort (CHD vs Myocardio vs Combined).

In [ ]:
# Compare ALL models: load mc_cv_model_metrics.csv from every variant and build summary table
import pandas as pd
from pathlib import Path

models_dir = Path("outputs/models")
if not models_dir.exists():
    models_dir = Path("outputs") / "models"
if not models_dir.exists():
    print("outputs/models not found. Run the training cell first.")
else:
    rows = []
    for variant_dir in sorted(models_dir.iterdir()):
        if not variant_dir.is_dir():
            continue
        metrics_file = variant_dir / "mc_cv_model_metrics.csv"
        if not metrics_file.exists():
            continue
        try:
            df = pd.read_csv(metrics_file)
        except Exception as e:
            print(f"  Skip {variant_dir.name}: {e}")
            continue
        if df.empty or "C_Index_Mean" not in df.columns:
            continue
        # Parse cohort and variant from dir name (e.g. CHD_top -> CHD, top)
        name = variant_dir.name
        for suffix in ["_base", "_enhanced", "_top", "_wisotzkey", "_FULL"]:
            if name.endswith(suffix):
                cohort = name[: -len(suffix)]
                variant = suffix.lstrip("_")
                break
        else:
            cohort, variant = name, "?"
        # Best model in this variant (by C-index, then AU-PRC)
        df = df.sort_values(by=["C_Index_Mean", "AU_PRC_Mean"], ascending=[False, False])
        best = df.iloc[0]
        rows.append({
            "Cohort": cohort,
            "Variant": variant,
            "Best_Model": best.get("Model", "?"),
            "C_Index_Mean": round(best["C_Index_Mean"], 4),
            "C_Index_CI": f"[{best.get('C_Index_CI_Lower', best['C_Index_Mean']):.3f}, {best.get('C_Index_CI_Upper', best['C_Index_Mean']):.3f}]" if pd.notna(best.get("C_Index_CI_Lower")) and pd.notna(best.get("C_Index_CI_Upper")) else "",
            "Recall_Mean": round(best.get("Recall_Mean", float("nan")), 4) if pd.notna(best.get("Recall_Mean")) else None,
            "AUC_Mean": round(best.get("AUC_Mean", float("nan")), 4) if pd.notna(best.get("AUC_Mean")) else None,
            "AU_PRC_Mean": round(best.get("AU_PRC_Mean", float("nan")), 4) if pd.notna(best.get("AU_PRC_Mean")) else None,
            "n_splits": int(best.get("n_splits", 0)),
        })
    if rows:
        compare_df = pd.DataFrame(rows)
        # Pivot C-index: rows = cohort, columns = variant (order: base, enhanced, top, wisotzkey, FULL)
        pivot_cindex = compare_df.pivot(index="Cohort", columns="Variant", values="C_Index_Mean")
        preferred = [v for v in ["base", "enhanced", "top", "wisotzkey", "FULL"] if v in pivot_cindex.columns]
        pivot_cindex = pivot_cindex[preferred + [c for c in pivot_cindex.columns if c not in preferred]]
        def _show(x):
            try:
                display(x)
            except NameError:
                print(x.to_string())
        print("C-index by Cohort × Variant (best model per variant):")
        _show(pivot_cindex)
        print("\nFull comparison (best model per variant):")
        _show(compare_df)
    else:
        print("No mc_cv_model_metrics.csv found under outputs/models/. Run the training cell first.")

In [8]:
# Check training results
import json

print("\nTraining Results Summary:")
print("-" * 80)

# All cohort models: top and Wisotzkey per cohort (CHD, Myocardio, Combined)
model_variants = [
    ("CHD (top)", "CHD_top"),
    ("CHD (Wisotzkey)", "CHD_wisotzkey"),
    ("Myocardio (top)", "Myocardio_top"),
    ("Myocardio (Wisotzkey)", "Myocardio_wisotzkey"),
    ("Combined (top)", "Combined_top"),
    ("Combined (Wisotzkey)", "Combined_wisotzkey"),
]

found_any = False
for variant_name, variant_dir in model_variants:
    best_model_file = CALCULATOR_DIR / "outputs" / "models" / variant_dir / "best_model.txt"
    if best_model_file.exists():
        found_any = True
        print(f"\n{variant_name} Model ({variant_dir}):")
        with open(best_model_file, 'r') as f:
            content = f.read()
            print(content)
            
            # Extract model performance info
            lines = content.split('\n')
            model_performances = {}
            for line in lines:
                if 'CatBoost C-index:' in line or 'CatBoost' in line and 'C-index' in line:
                    # Try to extract C-index value
                    parts = line.split(':')
                    if len(parts) > 1:
                        model_performances['CatBoost'] = parts[-1].strip().split()[0]
                elif 'XGBoost RF C-index:' in line or ('XGBoost RF' in line and 'C-index' in line):
                    parts = line.split(':')
                    if len(parts) > 1:
                        model_performances['XGBoost RF'] = parts[-1].strip().split()[0]
                elif 'XGBoost C-index:' in line or ('XGBoost' in line and 'C-index' in line and 'RF' not in line):
                    parts = line.split(':')
                    if len(parts) > 1:
                        model_performances['XGBoost'] = parts[-1].strip().split()[0]
            
            # Also try to extract from "Best Model (MC-CV):" line
            for line in lines:
                if 'Best Model (MC-CV):' in line:
                    best_model_name = line.split(':')[1].strip() if ':' in line else ''
                    # Try to find C-index for this model
                    for perf_line in lines:
                        if best_model_name in perf_line and 'C-index' in perf_line:
                            parts = perf_line.split(':')
                            if len(parts) > 1:
                                c_index_val = parts[-1].strip().split()[0]
                                if best_model_name not in model_performances:
                                    model_performances[best_model_name] = c_index_val
        
        if model_performances:
            print(f"\n  Model Performance Comparison:")
            for model_name, c_index in sorted(model_performances.items(), 
                                               key=lambda x: float(x[1]) if x[1].replace('.', '').isdigit() else 0, 
                                               reverse=True):
                print(f"    {model_name:20s} C-index: {c_index}")
        
        # List model files for this variant
        models_dir = CALCULATOR_DIR / "outputs" / "models" / variant_dir
        if models_dir.exists():
            catboost_file = models_dir / "catboost_model.cbm"
            xgboost_file = models_dir / "xgboost_model.ubj"
            xgboost_rf_file = models_dir / "xgboost_rf_model.ubj"
            
            print(f"\n  Trained Models ({variant_name}):")
            if catboost_file.exists():
                size_mb = catboost_file.stat().st_size / (1024 * 1024)
                print(f"    ✓ CatBoost: {catboost_file.name} ({size_mb:.2f} MB)")
            else:
                print(f"    ○ CatBoost: Not found")
            
            if xgboost_file.exists():
                size_mb = xgboost_file.stat().st_size / (1024 * 1024)
                print(f"    ✓ XGBoost: {xgboost_file.name} ({size_mb:.2f} MB)")
            else:
                print(f"    ○ XGBoost: Not found")
            
            if xgboost_rf_file.exists():
                size_mb = xgboost_rf_file.stat().st_size / (1024 * 1024)
                print(f"    ✓ XGBoost RF: {xgboost_rf_file.name} ({size_mb:.2f} MB)")
            else:
                print(f"    ○ XGBoost RF: Not found")
            
            # Check feature count
            feature_file = models_dir / "feature_names.json"
            if feature_file.exists():
                with open(feature_file, 'r') as f:
                    features = json.load(f)
                    print(f"\n  Feature Summary ({variant_name}):")
                    print(f"    Total features: {len(features)}")
                    print(f"    ✓ primary_etiology: {'primary_etiology' in features or any('primary_etiology' in f for f in features)}")
                    print(f"    ✓ vad_combined: {'vad_combined' in features}")
                    print(f"    ✓ vent_combined: {'vent_combined' in features}")
                    print(f"    ✓ ecmo_combined: {'ecmo_combined' in features}")
                    print(f"    ✓ donor_weight_ratio: {'donor_weight_ratio' in features}")
                    print(f"    ✓ donor_size_ratio: {'donor_size_ratio' in features}")
                    print(f"    ✓ chd_lat: {'chd_lat' in features}")

if not found_any:
    print(f"\n⚠ {COHORT}: Best model files not found")
    print(f"  Checked directories:")
    for variant_name, variant_dir in model_variants:
        checked_path = CALCULATOR_DIR / "outputs" / "models" / variant_dir / "best_model.txt"
        print(f"    - {variant_dir}/best_model.txt: {'✓ Found' if checked_path.exists() else '✗ Not found'}")


Training Results Summary:
--------------------------------------------------------------------------------

Top Model Model (Combined_top):
Best Model (MC-CV): XGBoost
Selection Criteria: C-index (primary), AU-PRC (tiebreaker)
Standard metrics (all models): C-index, Recall, AUC, AU-PRC

Best model MC-CV metrics:
  C-index: 0.639855 (95% CI: [0.606166, 0.673649], SD: 0.019135)
  Recall:  0.628478 ± 0.107876
  AUC:     0.614814 ± 0.022104
  AU-PRC:  0.235081 ± 0.019124
  n_splits: 25

Temporal Split Results:
  CatBoost: 0.580794
  XGBoost: 0.636556
  XGBoost RF: 0.628740

MC-CV Model Performance (all models) - C-index, Recall, AUC, AU-PRC:
  CatBoost:
    C-index: 0.542615 ± 0.076050 (95% CI: 0.357748 - 0.620126)
    Recall:  0.509565 ± 0.219330
    AUC:     0.543891 ± 0.045161
    AU-PRC:  0.193113 ± 0.022553
    [25 splits]
  XGBoost:
    C-index: 0.639855 ± 0.019135 (95% CI: 0.606166 - 0.673649)
    Recall:  0.628478 ± 0.107876
    AUC:     0.614814 ± 0.022104
    AU-PRC:  0.235081 ±

## 4. Causal Analysis Workflow

Generate SHAP values and extract causal factors using Formal Feature Attribution for the **top causal features model** (`Combined_top`).

Outputs go to `outputs/shap_ffa/Combined_top/` and are used by the dashboard (single model).

4. Top Model SHAP/FFA Analysis

Generate SHAP values and extract causal factors for the **Top Model** (`Combined_top`, top 15 causal features).

**Step 1: SHAP Value Computation**

The workflow checks which model is best (from `best_model.txt` in `Combined_top/`):

- **If XGBoost is best model:**
  - ✅ Computes SHAP values from **best XGBoost model** only
  - Uses simplified pipeline (XGBoost SHAP only)
  - No weights needed (single model)

- **If CatBoost is best model:**
  - ✅ Computes SHAP values from **best CatBoost model**
  - ✅ Computes SHAP values from **best XGBoost model**
  - ✅ **Automatically determines weights** based on C-index values:
    - Weights are calculated from relative C-index performance
    - CatBoost (best model) gets higher weight
    - XGBoost gets lower weight
    - Weights normalized to sum to 1.0
  - Uses combined pipeline (CatBoost + XGBoost SHAP)

**Step 2: FFA Rule Extraction**

- ✅ **Always uses**: Best XGBoost JSON model for rule extraction
  - Rules are extracted from XGBoost JSON structure (`*_final_model_xgboost.json`)
  - CatBoost JSON is **never** used for rule extraction (harder to parse due to categorical hashing)
  - Even if CatBoost is the best model, rules come from XGBoost JSON

**Step 3: Rule Filtering & Causal Responsibility**

- Rules are filtered using SHAP importance values (from Step 1)
- Causal responsibility is calculated as:
  ```
  causal_responsibility = (rule_frequency / total_rules) × SHAP_importance
  ```
- Where `SHAP_importance` comes from:
  - XGBoost SHAP only (if XGBoost is best), OR
  - Combined CatBoost + XGBoost SHAP (if CatBoost is best)

**Note:** FFA analysis is **REQUIRED**. The workflow requires `ffa_analysis/` directory with:
- `ffa_utils.py` (with `load_model_json`, `extract_feature_mappings`)
- `xgboost_axp_explainer.py` (with `XGBoostSymbolicExplainer`, `PathConfig`)

These modules have been restored from git history and are now available in the repository.

**Summary - Explicit Model Usage:**

| Component | Model Used | Condition |
|-----------|-----------|-----------|
| **SHAP Values** | Best XGBoost | Always |
| **SHAP Values** | Best CatBoost | Only if CatBoost is best model |
| **Rule Extraction** | Best XGBoost JSON | Always (regardless of which model is best) |
| **FFA Analysis** | Best XGBoost JSON + SHAP | Always |

**Key Point:** Even if CatBoost is the best model, the FFA analysis still uses the **XGBoost JSON model** for rule extraction, but filters those rules using **combined SHAP values** (CatBoost + XGBoost).

### 4. Top Model SHAP/FFA Analysis

In [9]:
# Run SHAP/FFA workflow for Top Model (Combined_top)
import subprocess

print(f"\n{'=' * 80}")
print("Running SHAP + FFA Analysis for Top Model")
print(f"{'=' * 80}")
print(f"Analyzing Top Combined model (Combined_top)...")
print("-" * 80)
print(f"\nCausal Analysis Process:")
print(f"  1. Check best model (from best_model.txt in Combined_top/)")
print(f"  2. Compute SHAP values:")
print(f"     - If XGBoost is best: XGBoost SHAP only")
print(f"     - If CatBoost is best: Combined SHAP (CatBoost + XGBoost)")
print(f"     - Weights auto-determined from C-index values")
print(f"  3. Extract rules from best XGBoost JSON model (if FFA available)")
print(f"  4. Filter rules using SHAP importance")
print(f"  5. Calculate causal responsibility (if FFA available)")
print(f"  6. Generate top {TOP_K} causal factors")
print("-" * 80)
print(f"\nModel Variant: Top (top 15 causal/importance features only)")
print(f"Output: outputs/shap_ffa/Combined_top/ (dashboard data for Risk Calculator and Causal Analysis tabs)")
print("-" * 80)
print(f"\nNote: If FFA modules are not available:")
print(f"  - Workflow continues with SHAP analysis only")
print(f"  - Uses SHAP importance instead of causal responsibility")
print(f"  - Dashboard will show SHAP-based rankings")
print("-" * 80)

# Cohorts to run SHAP/FFA for (all when COHORT is None; subprocess requires str, not None)
cohorts_for_shap = COHORTS if COHORT is None else [COHORT]
calc_dir = Path(CALCULATOR_DIR).resolve() if CALCULATOR_DIR is not None else Path().resolve()
if not Path(calc_dir).exists():
    raise ValueError("CALCULATOR_DIR not set or invalid. Run the path setup cell first.")

try:
    for cohort_name in cohorts_for_shap:
        cmd = [
            sys.executable,
            str(Path(calc_dir) / "run_shap_ffa_workflow.py"),
            "--cohort", cohort_name,
            "--model-variant", "top",
            "--top-k", str(TOP_K)
        ]
        if WEIGHT_CATBOOST is not None:
            cmd.extend(["--weight-catboost", str(WEIGHT_CATBOOST)])
        if WEIGHT_XGBOOST is not None:
            cmd.extend(["--weight-xgboost", str(WEIGHT_XGBOOST)])
        print(f"\n--- SHAP/FFA for cohort: {cohort_name} ---")
        result = subprocess.run(
            cmd,
            cwd=str(Path(calc_dir).resolve()),
            capture_output=False,
            text=True
        )
        if result.returncode != 0:
            print(f"\n⚠ SHAP/FFA for {cohort_name} exited with code: {result.returncode}")
            break

    if result.returncode == 0:
        print(f"\n✓ Top model SHAP/FFA analysis complete!")
        print(f"\nResults include:")
        print(f"  - Top {TOP_K} causal factors with causal responsibility scores")
        print(f"  - Feature importance rankings (SHAP-based)")
        print(f"  - Rule-based FFA analysis results")
        print(f"  - Dashboard data for Risk Calculator (outputs/shap_ffa/Combined_top/)")
        print(f"\nNote: Causal factors are calculated using:")
        print(f"  - Rules extracted from best XGBoost JSON model (Combined_top/)")
        print(f"  - SHAP importance from best models (XGBoost, and CatBoost if CatBoost is best)")
        print(f"  - Formula: causal_responsibility = (rule_frequency / total_rules) × SHAP_importance")
    else:
        print(f"\n⚠ SHAP/FFA exited with code: {result.returncode}")
except Exception as e:
    print(f"\n✗ Error running SHAP/FFA: {e}")
    logger.error(f"Error running SHAP/FFA", exc_info=True)

print(f"\n{'=' * 80}")
print("Top Model SHAP/FFA analysis complete!")
print(f"{'=' * 80}")

2026-02-16 02:20:07,105 - __main__ - ERROR - Error running SHAP/FFA
Traceback (most recent call last):
  File "/tmp/ipykernel_31937/827133901.py", line 45, in <module>
    result = subprocess.run(
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "/usr/local/lib/python3.11/subprocess.py", line 1885, in _execute_child
    self.pid = _fork_exec(
               ^^^^^^^^^^^
TypeError: expected str, bytes or os.PathLike object, not NoneType



Running SHAP + FFA Analysis for Top Model
Analyzing Top Combined model (Combined_top)...
--------------------------------------------------------------------------------

Causal Analysis Process:
  1. Check best model (from best_model.txt in Combined_top/)
  2. Compute SHAP values:
     - If XGBoost is best: XGBoost SHAP only
     - If CatBoost is best: Combined SHAP (CatBoost + XGBoost)
     - Weights auto-determined from C-index values
  3. Extract rules from best XGBoost JSON model (if FFA available)
  4. Filter rules using SHAP importance
  5. Calculate causal responsibility (if FFA available)
  6. Generate top 15 causal factors
--------------------------------------------------------------------------------

Model Variant: Top (top 15 causal/importance features only)
Output: outputs/shap_ffa/Combined_top/ (dashboard data for Risk Calculator and Causal Analysis tabs)
--------------------------------------------------------------------------------

Note: If FFA modules are not avai

2026-02-10 03:19:27,527 - __main__ - INFO - FFA modules loaded using direct file import
2026-02-10 03:19:27,792 - botocore.credentials - INFO - Found credentials from IAM Role: EC2_Spot
2026-02-10 03:19:27,834 - __main__ - INFO - Using model variant: top -> model directory: Combined_top
2026-02-10 03:19:27,834 - __main__ - INFO - Output directory: /home/pgx3874/phts/graft-loss/cohort_analysis/calculator/outputs/shap_ffa/Combined_top
2026-02-10 03:19:27,834 - __main__ - INFO - Best model from file: XGBoost
2026-02-10 03:19:27,834 - __main__ - INFO - ================================================================================
2026-02-10 03:19:27,834 - __main__ - INFO - SHAP + FFA Analysis for Combined Cohort
2026-02-10 03:19:27,834 - __main__ - INFO - ================================================================================
2026-02-10 03:19:27,834 - __main__ - INFO - Best model: XGBoost - Using simplified pipeline (XGBoost only)
2026-02-10 03:19:27,834 - __main__ - INFO - Stra


✓ Top model SHAP/FFA analysis complete!

Results include:
  - Top 15 causal factors with causal responsibility scores
  - Feature importance rankings (SHAP-based)
  - Rule-based FFA analysis results
  - Dashboard data for Risk Calculator (outputs/shap_ffa/Combined_top/)

Note: Causal factors are calculated using:
  - Rules extracted from best XGBoost JSON model (Combined_top/)
  - SHAP importance from best models (XGBoost, and CatBoost if CatBoost is best)
  - Formula: causal_responsibility = (rule_frequency / total_rules) × SHAP_importance

Top Model SHAP/FFA analysis complete!



✓ Baseline model SHAP/FFA analysis complete!

Results include:
  - Top 10 causal factors with causal responsibility scores
  - Feature importance rankings (SHAP-based)
  - Rule-based FFA analysis results
  - Dashboard data for Baseline Model tab (outputs/shap_ffa/Combined_base/)

Note: Causal factors are calculated using:
  - Rules extracted from best XGBoost JSON model (Combined_base/)
  - SHAP importance from best models (XGBoost, and CatBoost if CatBoost is best)
  - Formula: causal_responsibility = (rule_frequency / total_rules) × SHAP_importance

Baseline Model SHAP/FFA analysis complete!


## 5. Inspect Results

View top causal factors, feature importance, and dashboard data for the Combined model.

In [ ]:
# Load and display dashboard data for the top causal features model
import json
import pandas as pd

print("\n" + "=" * 80)
print("Results Summary - Top Model (Combined_top)")
print("=" * 80)

# Single model: top 15 causal features only
model_variants = [
    ("Top Model", "Combined_top")
]

for variant_name, variant_dir in model_variants:
    print(f"\n{'=' * 80}")
    print(f"{variant_name} Model ({variant_dir}) - Top {TOP_K} Causal Factors")
    print("=" * 80)
    
    dashboard_data_file = (
        CALCULATOR_DIR / "outputs" / "shap_ffa" / variant_dir / "dashboard_data.json"
    )
    
    if dashboard_data_file.exists():
        with open(dashboard_data_file, 'r') as f:
            dashboard_data = json.load(f)
        
        top_factors = dashboard_data.get('top_causal_factors', [])[:TOP_K]
        
        if top_factors:
            for idx, factor in enumerate(top_factors, 1):
                importance = factor.get('causal_responsibility', 
                                     factor.get('importance', 
                                               factor.get('combined_importance_norm', 0)))
                print(f"{idx:2d}. {factor['feature']:40s} "
                      f"(Importance: {importance:.4f})")
        else:
            print("  (No causal factors available)")
        
        # Display summary statistics
        if 'summary' in dashboard_data:
            print(f"\n  Summary Statistics:")
            summary = dashboard_data['summary']
            for key, value in summary.items():
                print(f"    {key}: {value}")
        
        # Check for key features in top factors
        print(f"\n  Key Features Check:")
        top_feature_names = [f['feature'] for f in top_factors]
        key_features = {
            'primary_etiology': any('primary_etiology' in f for f in top_feature_names),
            'vad_combined': 'vad_combined' in top_feature_names,
            'vent_combined': 'vent_combined' in top_feature_names,
            'ecmo_combined': 'ecmo_combined' in top_feature_names,
            'donor_weight_ratio': 'donor_weight_ratio' in top_feature_names,
            'donor_size_ratio': 'donor_size_ratio' in top_feature_names,
            'chd_lat': 'chd_lat' in top_feature_names,
            'egfr_tx': 'egfr_tx' in top_feature_names or any('egfr' in f for f in top_feature_names),
            'txfcpra': 'txfcpra' in top_feature_names,
            'hxsurg': 'hxsurg' in top_feature_names
        }
        for feature, present in key_features.items():
            status = "✓" if present else "○"
            print(f"    {status} {feature}")
    else:
        print(f"\n⚠ Dashboard data not found for {variant_name} model")
        print(f"  Expected: {dashboard_data_file}")
        print("  Run SHAP/FFA analysis for top model first (Section 4)")

### a. Feature Importances

In [ ]:
# Load and display feature importance for the top causal features model
print("\n" + "=" * 80)
print("Feature Importance Rankings - Top Model (Combined_top)")
print("=" * 80)

# Single model: top 15 causal features only
model_variants = [
    ("Top Model", "Combined_top")
]

for variant_name, variant_dir in model_variants:
    print(f"\n{'=' * 80}")
    print(f"{variant_name} Model ({variant_dir}) - Feature Importance")
    print("=" * 80)
    
    # Check variant dir first, then fallback to COHORT (e.g. Combined) if no files
    models_root = CALCULATOR_DIR / "outputs" / "models"
    variant_path = models_root / variant_dir
    fallback_path = models_root / COHORT if COHORT != variant_dir else None
    importance_files = list(variant_path.glob("importance_*.csv")) if variant_path.exists() else []
    if not importance_files:
        importance_files = list(variant_path.glob("mc_cv_*_feature_importance.csv")) if variant_path.exists() else []
    if not importance_files and fallback_path and fallback_path.exists():
        importance_files = list(fallback_path.glob("importance_*.csv")) + list(fallback_path.glob("mc_cv_*_feature_importance.csv"))
    
    if importance_files:
        for imp_file in sorted(importance_files):
            model_name = imp_file.stem.replace(f"importance_{variant_dir}_", "").replace("mc_cv_", "").replace("_feature_importance", "")
            print(f"\n  {model_name}:")
            df = pd.read_csv(imp_file)
            imp_col = "importance" if "importance" in df.columns else "importance_mean"
            print(f"    Total features: {len(df)}")
            print(f"    Top 10 features:")
            top10 = df.nlargest(10, imp_col)
            for idx, row in top10.iterrows():
                print(f"      {row['feature']:40s} {row[imp_col]:.4f}")
            
            # Check for key features
            feature_list = df['feature'].tolist()
            print(f"\n    Key Features Status:")
            key_features = {
                'primary_etiology': any('primary_etiology' in f for f in feature_list),
                'vad_combined': 'vad_combined' in feature_list,
                'vent_combined': 'vent_combined' in feature_list,
                'ecmo_combined': 'ecmo_combined' in feature_list,
                'donor_weight_ratio': 'donor_weight_ratio' in feature_list,
                'donor_size_ratio': 'donor_size_ratio' in feature_list,
                'chd_lat': 'chd_lat' in feature_list,
                'egfr_tx': 'egfr_tx' in feature_list,
                'txfcpra': 'txfcpra' in feature_list,
                'hxsurg': 'hxsurg' in feature_list
            }
            for feature, present in key_features.items():
                status = "✓" if present else "○"
                if present:
                    rank = df[df['feature'] == feature].index[0] + 1 if feature in feature_list else "N/A"
                    print(f"      {status} {feature:25s} (Rank: {rank})")
                else:
                    print(f"      {status} {feature:25s} (Not found)")
    else:
        print(f"\n⚠ No feature importance files found for {variant_name} model")
        print("  Train top model first (Section 1) and run SHAP/FFA (Section 4)")

### b. Visualizations

Create visualization of top causal factors for the top model (Combined_top).

In [ ]:
# Plot top causal factors for the top model (Combined_top)
try:
    import matplotlib.pyplot as plt
    import numpy as np
    
    # Load dashboard data for top model only
    top_dashboard_file = (
        CALCULATOR_DIR / "outputs" / "shap_ffa" / "Combined_top" / "dashboard_data.json"
    )
    top_data = None
    
    if top_dashboard_file.exists():
        with open(top_dashboard_file, 'r') as f:
            top_data = json.load(f)
        print(f"\n✓ Loaded Top model data from: {top_dashboard_file}")
    else:
        print(f"\n⚠ Top model dashboard data not found: {top_dashboard_file}")
        print("  Run SHAP/FFA analysis for top model first (Section 4)")
    
    if top_data:
        top_factors = top_data.get('top_causal_factors', [])[:TOP_K]
        if top_factors:
            features = [f['feature'] for f in top_factors]
            importance = [f.get('causal_responsibility', 
                              f.get('importance', 
                                   f.get('combined_importance_norm', 0))) 
                        for f in top_factors]
            
            plt.figure(figsize=(10, max(6, len(features) * 0.4)))
            plt.barh(range(len(features)), importance, color='#3b82f6', alpha=0.8)
            plt.yticks(range(len(features)), features)
            plt.xlabel('Causal Responsibility / Importance')
            plt.title(f'Top {TOP_K} Causal Factors - Top Model (Combined_top)')
            plt.gca().invert_yaxis()
            plt.tight_layout()
            
            plot_file = CALCULATOR_DIR / "outputs" / "shap_ffa" / "Combined_top" / f"top_{TOP_K}_factors.png"
            plt.savefig(plot_file, dpi=150, bbox_inches='tight')
            print(f"\n✓ Saved plot: {plot_file}")
            plt.show()
        else:
            print("\n⚠ No causal factors available in dashboard data")
    else:
        print("\n⚠ No dashboard data available for visualization")
        print("  Run SHAP/FFA analysis first (Section 4)")
            
except ImportError:
    print("\n⚠ Matplotlib not available. Skipping visualizations.")
    print("  Install with: pip install matplotlib")
except Exception as e:
    print(f"\n⚠ Error creating visualizations: {e}")
    import traceback
    traceback.print_exc()

### c. Export Summary

Create a summary JSON file with all results for the top model (Combined_top).

In [ ]:
# Create workflow summary
from datetime import datetime

summary = {
    "workflow": "Calculator Model Training + SHAP/FFA Analysis",
    "model_strategy": "Single top 15 causal features model (Combined_top)",
    "timestamp": datetime.now().isoformat(),
    "configuration": {
        "cohort": COHORT,
        "top_k": TOP_K,
        "weight_catboost": WEIGHT_CATBOOST,
        "weight_xgboost": WEIGHT_XGBOOST,
        "debug_mode": DEBUG_MODE
    },
    "models": {
        "top": {}
    }
}

# Single model: top 15 causal features only
model_variants = [
    ("top", "Combined_top")
]

for variant_name, variant_dir in model_variants:
    variant_summary = {}
    
    # Best model
    best_model_file = CALCULATOR_DIR / "outputs" / "models" / variant_dir / "best_model.txt"
if best_model_file.exists():
    with open(best_model_file, 'r') as f:
        content = f.read()
        lines = content.split('\n')
        for line in lines:
            if line.startswith("Best Model:"):
                variant_summary["best_model"] = line.replace("Best Model: ", "").strip()
            elif line.startswith("C-index:"):
                try:
                    variant_summary["c_index"] = float(line.replace("C-index: ", "").strip())
                except:
                    pass
    
    # Dashboard data
    dashboard_file = CALCULATOR_DIR / "outputs" / "shap_ffa" / variant_dir / "dashboard_data.json"
if dashboard_file.exists():
    with open(dashboard_file, 'r') as f:
        dashboard_data = json.load(f)
        variant_summary["top_factors_count"] = len(dashboard_data.get('top_causal_factors', []))
        if dashboard_data.get('top_causal_factors'):
            variant_summary["top_factor"] = dashboard_data['top_causal_factors'][0]['feature']
            variant_summary["top_factor_importance"] = dashboard_data['top_causal_factors'][0].get(
                'causal_responsibility', 
                dashboard_data['top_causal_factors'][0].get('importance', 0)
            )
        
        # List top 5 factors
        top5 = dashboard_data.get('top_causal_factors', [])[:5]
        variant_summary["top_5_factors"] = [
            {
                "feature": f['feature'],
                "importance": f.get('causal_responsibility', 
                                  f.get('importance', 
                                       f.get('combined_importance_norm', 0)))
            }
            for f in top5
        ]
    
    # Feature count
    feature_file = CALCULATOR_DIR / "outputs" / "models" / variant_dir / "feature_names.json"
    if feature_file.exists():
        with open(feature_file, 'r') as f:
            features = json.load(f)
            variant_summary["total_features"] = len(features)
            variant_summary["key_features"] = {
            "primary_etiology": 'primary_etiology' in features or any('primary_etiology' in f for f in features),
            "vad_combined": 'vad_combined' in features,
            "vent_combined": 'vent_combined' in features,
            "ecmo_combined": 'ecmo_combined' in features,
            "donor_weight_ratio": 'donor_weight_ratio' in features,
            "donor_size_ratio": 'donor_size_ratio' in features,
            "chd_lat": 'chd_lat' in features,
            "egfr_tx": 'egfr_tx' in features,
            "txfcpra": 'txfcpra' in features,
            "hxsurg": 'hxsurg' in features
        }
    
    summary["models"][variant_name] = variant_summary

# Save summary
summary_file = CALCULATOR_DIR / "outputs" / "workflow_summary.json"
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Workflow summary saved to: {summary_file}")
print("\nSummary:")
print(json.dumps(summary, indent=2))

## 6. Deploy Risk Calculator (Lambda + S3)

Deploy the trained models and dashboard to AWS Lambda and S3 for production use.

### Deployment Overview

**Components:**
- **AWS Lambda**: Container-based function with models baked in
- **API Gateway**: REST API endpoints for risk calculation
- **S3**: Static HTML dashboard hosting

**Prerequisites:**
- AWS CLI configured with appropriate permissions
- Docker installed and running
- Models trained (Section 3)
- SHAP/FFA analysis complete (Section 4)
- Risk distributions computed (if needed)

### Deployment Steps

1. **Prepare Lambda Directory**: Run `prepare_lambda_dir_phts.py` (copies both *_top and *_wisotzkey models, per-cohort deployed_variant file, and dashboard data; merges aggregated feature importance)
2. **Build and Push**: Run `./docker_build_phts.sh` — builds image, pushes to ECR, and updates Lambda by default
3. **Setup API Gateway**: Run `./setup_api_gateway.sh` (includes GET /metadata, GET /model-metrics, POST /risk, POST /causal)
4. **Upload HTML**: Deploy dashboard HTML to S3

In [ ]:
# Step 1: Prepare Lambda Directory (Idempotent - only updates if needed)
import subprocess
from pathlib import Path
import os

print(f"\n{'=' * 80}")
print("Step 1: Preparing Lambda Directory")
print(f"{'=' * 80}")

risk_dashboard_dir = CALCULATOR_DIR / "risk_dashboard"
prepare_script = risk_dashboard_dir / "prepare_lambda_dir_phts.py"
lambda_dir = risk_dashboard_dir / "lambda_dir_phts"

# Check if Lambda directory already exists and is up to date (example: Combined_top; prepare copies all cohorts)
MODEL_DIR_NAME = "Combined_top"
needs_prepare = True
if lambda_dir.exists():
    # Check if models exist and are recent
    models_dir = lambda_dir / "models" / MODEL_DIR_NAME
    dashboard_dir = lambda_dir / "dashboard_data" / MODEL_DIR_NAME
    
    # Check if source models are newer than lambda_dir models
    source_models_dir = CALCULATOR_DIR / "outputs" / "models" / MODEL_DIR_NAME
    source_dashboard_dir = CALCULATOR_DIR / "outputs" / "shap_ffa" / MODEL_DIR_NAME
    
    if models_dir.exists() and source_models_dir.exists():
        # Get most recent model file modification time
        source_model_files = list(source_models_dir.glob("*.cbm")) + list(source_models_dir.glob("*.ubj"))
        lambda_model_files = list(models_dir.glob("*.cbm")) + list(models_dir.glob("*.ubj"))
        
        if source_model_files and lambda_model_files:
            source_mtime = max(f.stat().st_mtime for f in source_model_files)
            lambda_mtime = max(f.stat().st_mtime for f in lambda_model_files)
            
            if source_mtime <= lambda_mtime:
                print(f"\n✓ Lambda directory is up to date")
                print(f"  Source models: {len(source_model_files)} files")
                print(f"  Lambda models: {len(lambda_model_files)} files")
                print(f"  Last update: {lambda_mtime}")
                needs_prepare = False

if needs_prepare and prepare_script.exists():
    print(f"\nRunning: {prepare_script}")
    print("This will copy/update models, dashboard data, and risk distributions to lambda_dir_phts/")
    print("-" * 80)
    
    try:
        result = subprocess.run(
            [sys.executable, str(prepare_script)],
            cwd=str(risk_dashboard_dir),
            capture_output=False,
            text=True
        )
        
        if result.returncode == 0:
            print(f"\n✓ Lambda directory prepared successfully!")
            
            # Check what was created/updated
            if lambda_dir.exists():
                print(f"\n  Lambda directory structure:")
                print(f"    {lambda_dir}")
                
                # Check models
                models_dir = lambda_dir / "models" / MODEL_DIR_NAME
                if models_dir.exists():
                    model_files = list(models_dir.glob("*.cbm")) + list(models_dir.glob("*.ubj"))
                    print(f"    ✓ Models: {len(model_files)} files")
                
                # Check dashboard data
                dashboard_dir = lambda_dir / "dashboard_data" / MODEL_DIR_NAME
                if dashboard_dir.exists():
                    dashboard_files = list(dashboard_dir.glob("*.json")) + list(dashboard_dir.glob("*.csv"))
                    print(f"    ✓ Dashboard data: {len(dashboard_files)} files")
                
                # Check risk distributions
                risk_dist_dir = lambda_dir / "risk_distributions"
                if risk_dist_dir.exists():
                    risk_files = list(risk_dist_dir.glob("*.json"))
                    print(f"    ✓ Risk distributions: {len(risk_files)} files")
        else:
            print(f"\n⚠ Script exited with code: {result.returncode}")
    except Exception as e:
        print(f"\n✗ Error preparing Lambda directory: {e}")
        logger.error("Error preparing Lambda directory", exc_info=True)
elif not prepare_script.exists():
    print(f"\n⚠ Prepare script not found: {prepare_script}")
    print("  Expected location: risk_dashboard/prepare_lambda_dir_phts.py")

print(f"\n{'=' * 80}")

In [ ]:
# Step 2: Build and Push Docker Image (Idempotent - only if Lambda dir changed)
# Set to True to always run build (e.g. after Lambda code changes)
FORCE_DOCKER_BUILD = False

print(f"\n{'=' * 80}")
print("Step 2: Build and Push Docker Image")
print(f"{'=' * 80}")

docker_script = risk_dashboard_dir / "docker_build_phts.sh"

# Rebuild if lambda_dir was just updated, or if user set FORCE_DOCKER_BUILD
needs_docker_build = needs_prepare or FORCE_DOCKER_BUILD

if docker_script.exists():
    print(f"\nDocker build strategy:")
    print(f"  - Lambda directory was {'updated' if needs_prepare else 'unchanged'}")
    print(f"  - Docker image will be {'built' if needs_docker_build else 'skipped'}")
    print("-" * 80)
    print("\nThis will:")
    print("  1. Build Docker image with models and dependencies")
    print("  2. Push image to AWS ECR (Elastic Container Registry)")
    print("-" * 80)
    print("\n⚠ Note: This requires:")
    print("  - Docker installed and running")
    print("  - AWS CLI configured with ECR permissions")
    print("  - AWS credentials with push access to ECR")
    print("-" * 80)
    
    if needs_docker_build:
        response = input("\nProceed with Docker build? (y/n): ").strip().lower()
    else:
        print("\n⏭ Skipping Docker build (Lambda directory unchanged)")
        print("  To rebuild: set FORCE_DOCKER_BUILD = True above and re-run this cell, or run from terminal:")
        print("  cd risk_dashboard && ./docker_build_phts.sh")
        response = 'n'
    
    if response == 'y':
        try:
            result = subprocess.run(
                ["bash", str(docker_script)],
                cwd=str(risk_dashboard_dir),
                capture_output=False,
                text=True
            )
            
            if result.returncode == 0:
                print(f"\n✓ Docker image built and pushed successfully!")
                print(f"\n  Next: Get ECR URI from output above and use it to update Lambda")
            else:
                print(f"\n⚠ Docker build exited with code: {result.returncode}")
        except Exception as e:
            print(f"\n✗ Error building Docker image: {e}")
            logger.error("Error building Docker image", exc_info=True)
else:
    print(f"\n⚠ Docker build script not found: {docker_script}")
    print("  Expected location: risk_dashboard/docker_build_phts.sh")

print(f"\n{'=' * 80}")


Proceed with Docker build? (y/n):  y


PHTS Lambda Container Build & Deploy

✓ Lambda directory already prepared

Validating lambda directory...
✓ Found 15 model files

Checking Docker permissions...
✓ Docker access verified

Building Docker image...


#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile.phts
#1 transferring dockerfile: 1.65kB done
#1 DONE 0.0s

#2 [internal] load metadata for public.ecr.aws/lambda/python:3.11
#2 DONE 0.0s

#3 [internal] load .dockerignore
#3 transferring context: 715B done
#3 DONE 0.0s

#4 [ 1/10] FROM public.ecr.aws/lambda/python:3.11
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 82.61kB done
#5 DONE 0.0s

#6 [ 6/10] COPY lambda_dir_phts/models/ /var/task/models/
#6 CACHED

#7 [ 2/10] RUN yum install -y gcc gcc-c++ cmake make &&     yum clean all &&     rm -rf /var/cache/yum
#7 CACHED

#8 [ 4/10] RUN pip install --no-cache-dir --upgrade pip setuptools wheel &&     pip install --no-cache-dir --prefer-binary -r /var/task/requirements.txt -t /var/task
#8 CACHED

#9 [ 3/10] COPY phts_requirements.txt /var/task/requirements.txt
#9 CACHED

#10 [ 5/10] COPY phts_lambda_function.py /var/task
#10 CACHED

#11 [ 7/10] COPY lambd

✓ Docker image built successfully

Logging in to ECR...


WARNING! Your password will be stored unencrypted in /home/pgx3874/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded
✓ Logged in to ECR

Checking ECR repository...
✓ ECR repository exists: phts-risk-calculator

Tagging image...
✓ Image tagged: 535362115856.dkr.ecr.us-east-1.amazonaws.com/phts-risk-calculator:latest

Pushing image to ECR (this may take a while)...
The push refers to repository [535362115856.dkr.ecr.us-east-1.amazonaws.com/phts-risk-calculator]
5f70bf18a086: Preparing
62ef85c7edc5: Preparing
e6e96c0e8b7a: Preparing
c99984b24ef1: Preparing
b89277e6b283: Preparing
ba35d0e09cd3: Preparing
2233004a4174: Preparing
0423d1326626: Preparing
bbc96e9e94c1: Preparing
bf8f2f1f8784: Preparing
91b133ae45ba: Preparing
9d2e3efe717a: Preparing
dbda07e25d10: Preparing
79e0b983c4ac: Preparing
5dcdfe5a68eb: Preparing
91b133ae45ba: Waiting
0423d1326626: Waiting
bbc96e9e94c1: Waiting
9d2e3efe717a: Waiting
bf8f2f1f8784: Waiting
dbda07e25d10: Waiting
79e0b983c4ac: Waiting
ba35d0e09cd3: Waiting
5dcdfe5a68eb: Waiting
2233004a4174: Waiting
5f70bf18a086: Layer already exists
b89277e6b283: Layer al

In [ ]:
# Step 3: Update Lambda Function (optional — docker_build_phts.sh now updates Lambda by default)
print(f"\n{'=' * 80}")
print("Step 3: Update Lambda Function")
print(f"{'=' * 80}")
print("Note: ./docker_build_phts.sh now runs update-function-code by default. Use this step if you built elsewhere or set UPDATE_LAMBDA=false.")

lambda_function_name = "phts-risk-calculator"
region = "us-east-1"

# Check if Lambda function exists
lambda_exists = False
try:
    result = subprocess.run(
        ["aws", "lambda", "get-function", "--function-name", lambda_function_name, "--region", region],
        capture_output=True,
        text=True,
        timeout=5
    )
    if result.returncode == 0:
        lambda_exists = True
        print(f"\n✓ Lambda function exists: {lambda_function_name}")
except:
    print(f"\n⚠ Could not check Lambda function status")
    print("  (AWS CLI may not be configured)")

if lambda_exists and needs_docker_build:
    print(f"\nLambda function will be updated with new Docker image")
    print("-" * 80)
    print("\n⚠ Note: AWS credentials are automatically used from EC2 instance role")
    print("  (No need to run 'aws configure' if instance has IAM role attached)")
    print("-" * 80)
    
    # Get AWS account ID and ECR URI
    try:
        result = subprocess.run(
            ["aws", "sts", "get-caller-identity", "--query", "Account", "--output", "text"],
            capture_output=True,
            text=True,
            timeout=5
        )
        if result.returncode == 0:
            account_id = result.stdout.strip()
            ecr_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/phts-risk-calculator:latest"
            print(f"\n  AWS Account ID: {account_id}")
            print(f"  ECR URI: {ecr_uri}")
            
            # Also get identity to show which role is being used
            identity_result = subprocess.run(
                ["aws", "sts", "get-caller-identity", "--output", "json"],
                capture_output=True,
                text=True,
                timeout=5
            )
            if identity_result.returncode == 0:
                import json
                identity = json.loads(identity_result.stdout)
                if "Arn" in identity:
                    print(f"  Using IAM Role: {identity['Arn']}")
            
            response = input("\nProceed with Lambda update? (y/n): ").strip().lower()
            
            if response == 'y':
                try:
                    result = subprocess.run(
                        [
                            "aws", "lambda", "update-function-code",
                            "--function-name", lambda_function_name,
                            "--image-uri", ecr_uri,
                            "--region", region
                        ],
                        capture_output=False,
                        text=True
                    )
                    
                    if result.returncode == 0:
                        print(f"\n✓ Lambda function updated successfully!")
                        print(f"\n  Waiting for update to complete...")
                        # Wait for function to be ready
                        subprocess.run(
                            ["aws", "lambda", "wait", "function-updated",
                             "--function-name", lambda_function_name,
                             "--region", region],
                            capture_output=True
                        )
                        print(f"  ✓ Lambda function is ready")
                    else:
                        print(f"\n⚠ Lambda update exited with code: {result.returncode}")
                except Exception as e:
                    print(f"\n✗ Error updating Lambda: {e}")
                    logger.error("Error updating Lambda", exc_info=True)
            else:
                print("\n⏭ Skipping Lambda update")
        else:
            print("\n⚠ Could not retrieve AWS Account ID")
    except:
        print("\n⚠ Could not retrieve AWS Account ID - check AWS CLI configuration")
elif not lambda_exists:
    print(f"\n⚠ Lambda function '{lambda_function_name}' does not exist")
    print("  Create it first using AWS Console or CLI")
elif not needs_docker_build:
    print(f"\n⏭ Skipping Lambda update (Docker image unchanged)")
else:
    print("\nTo update Lambda function manually:")
    print("-" * 80)
    print("\n1. Get your AWS Account ID:")
    print("   AWS_ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)")
    print("\n2. Construct ECR URI:")
    print("   ECR_URI=\"${AWS_ACCOUNT_ID}.dkr.ecr.us-east-1.amazonaws.com/phts-risk-calculator:latest\"")
    print("\n3. Update Lambda function:")
    print("   aws lambda update-function-code \\")
    print("       --function-name phts-risk-calculator \\")
    print("       --image-uri ${ECR_URI} \\")
    print("       --region us-east-1")

print(f"\n{'=' * 80}")


Proceed with Lambda update? (y/n):  y


{
    "FunctionName": "phts-risk-calculator",
    "FunctionArn": "arn:aws:lambda:us-east-1:535362115856:function:phts-risk-calculator",
    "Role": "arn:aws:iam::535362115856:role/phts-lambda-role",
    "CodeSize": 0,
    "Description": "",
    "Timeout": 60,
    "MemorySize": 3008,
    "LastModified": "2026-02-03T17:24:48.000+0000",
    "CodeSha256": "8da874b28691a3281b7c2f8268114591512e0f8768d7c72eb31b23c5e938acce",
    "Version": "$LATEST",
    "Environment": {
        "Variables": {
            "PHTS_BUCKET": "jerome-dixon.io",
            "S3_PREFIX": "uva/phts-risk-calculator",
            "API_GATEWAY_URL": "https://359vxflbzj.execute-api.us-east-1.amazonaws.com/prod"
        }
    },
    "TracingConfig": {
        "Mode": "PassThrough"
    },
    "RevisionId": "0d567021-1e5b-4634-8fcf-b417ef490f94",
    "State": "Active",
    "LastUpdateStatus": "InProgress",
    "LastUpdateStatusReason": "The function is being created.",
    "LastUpdateStatusReasonCode": "Creating",
    "Packa

In [ ]:
# Step 4: Verify API Gateway (Idempotent - assumes already set up)
print(f"\n{'=' * 80}")
print("Step 4: Verify API Gateway")
print(f"{'=' * 80}")

print("\n⚠ Note: API Gateway should already be set up")
print("  This step only verifies the configuration")
print("-" * 80)

# Check if API Gateway exists by trying to list APIs
try:
    result = subprocess.run(
        ["aws", "apigateway", "get-rest-apis", "--query", "items[?name=='phts-calculator-api'].id", "--output", "text"],
        capture_output=True,
        text=True,
        timeout=5
    )
    
    if result.returncode == 0 and result.stdout.strip():
        api_id = result.stdout.strip()
        api_url = f"https://{api_id}.execute-api.us-east-1.amazonaws.com/prod"
        print(f"\n✓ API Gateway found:")
        print(f"  API ID: {api_id}")
        print(f"  API URL: {api_url}")
        
        # Test metadata endpoint
        print(f"\n  Testing /metadata endpoint...")
        try:
            test_result = subprocess.run(
                ["curl", "-s", f"{api_url}/metadata?cohort={COHORT}"],
                capture_output=True,
                text=True,
                timeout=5
            )
            if test_result.returncode == 0:
                print(f"  ✓ API Gateway is responding")
            else:
                print(f"  ⚠ API Gateway may not be responding correctly")
        except:
            print(f"  ⚠ Could not test API endpoint (curl may not be available)")
    else:
        print(f"\n⚠ API Gateway not found or AWS CLI not configured")
        print("  If API Gateway needs to be set up, run:")
        print(f"    cd {risk_dashboard_dir}")
        print(f"    ./setup_api_gateway.sh")
except:
    print(f"\n⚠ Could not verify API Gateway (AWS CLI may not be configured)")
    print("  Assuming API Gateway is already set up")

print(f"\n{'=' * 80}")

In [ ]:
# Step 5: Upload HTML to S3 (Idempotent - only if HTML changed)
print(f"\n{'=' * 80}")
print("Step 5: Upload HTML Dashboard to S3")
print(f"{'=' * 80}")

html_file = risk_dashboard_dir / "phts_dashboard.html"
s3_bucket = "jerome-dixon.io"  # Update if different
s3_prefix = "uva/phts-risk-calculator"
s3_path = f"s3://{s3_bucket}/{s3_prefix}/index.html"

if html_file.exists():
    print(f"\nHTML file found: {html_file}")
    
    # Check if S3 file exists and compare modification times
    needs_upload = True
    try:
        result = subprocess.run(
            ["aws", "s3", "ls", s3_path, "--region", "us-east-1"],
            capture_output=True,
            text=True,
            timeout=5
        )
        
        if result.returncode == 0 and result.stdout.strip():
            # S3 file exists - check if local is newer
            local_mtime = html_file.stat().st_mtime
            
            # Parse S3 last modified time from ls output
            # Format: "2026-01-26 10:30:45    12345 index.html"
            s3_output = result.stdout.strip()
            if s3_output:
                print(f"\n✓ S3 file exists")
                print(f"  Checking if local file is newer...")
                
                # Get S3 file metadata
                head_result = subprocess.run(
                    ["aws", "s3api", "head-object", "--bucket", s3_bucket, 
                     "--key", f"{s3_prefix}/index.html", "--region", "us-east-1"],
                    capture_output=True,
                    text=True,
                    timeout=5
                )
                
                if head_result.returncode == 0:
                    import json
                    s3_meta = json.loads(head_result.stdout)
                    s3_mtime_str = s3_meta.get("LastModified", "")
                    if s3_mtime_str:
                        from datetime import datetime
                        s3_mtime = datetime.fromisoformat(s3_mtime_str.replace("Z", "+00:00")).timestamp()
                        
                        if local_mtime <= s3_mtime:
                            print(f"  ✓ S3 file is up to date (local: {local_mtime}, S3: {s3_mtime})")
                            needs_upload = False
                        else:
                            print(f"  ⚠ Local file is newer - will upload")
        else:
            print(f"\n⚠ S3 file not found - will upload")
    except:
        print(f"\n⚠ Could not check S3 file status (AWS CLI may not be configured)")
        print(f"  Will attempt upload")
    
    if needs_upload:
        print(f"\nUploading to S3:")
        print("-" * 80)
        print(f"  Source: {html_file}")
        print(f"  Destination: {s3_path}")
        print("-" * 80)
        print("\n⚠ Note: This requires:")
        print("  - AWS CLI configured")
        print("  - S3 write permissions")
        print("  - Bucket exists and is accessible")
        print("-" * 80)
        
        response = input("\nProceed with S3 upload? (y/n): ").strip().lower()
        
        if response == 'y':
            try:
                result = subprocess.run(
                    [
                        "aws", "s3", "cp",
                        str(html_file),
                        s3_path,
                        "--content-type", "text/html",
                        "--cache-control", "no-cache",
                        "--region", "us-east-1"
                    ],
                    capture_output=False,
                    text=True
                )
                
                if result.returncode == 0:
                    print(f"\n✓ HTML uploaded successfully to S3!")
                    print(f"\n  Dashboard URL: https://{s3_bucket}/{s3_prefix}/")
                else:
                    print(f"\n⚠ S3 upload exited with code: {result.returncode}")
            except Exception as e:
                print(f"\n✗ Error uploading to S3: {e}")
                logger.error("Error uploading to S3", exc_info=True)
        else:
            print("\n⏭ Skipping S3 upload")
    else:
        print(f"\n⏭ Skipping S3 upload (file is up to date)")
else:
    print(f"\n⚠ HTML file not found: {html_file}")
    print("  Expected location: risk_dashboard/phts_dashboard.html")

print(f"\n{'=' * 80}")


Proceed with S3 upload? (y/n):  y


upload: risk_dashboard/phts_dashboard.html to s3://jerome-dixon.io/uva/phts-risk-calculator/index.html

✓ HTML uploaded successfully to S3!

  Dashboard URL: https://jerome-dixon.io/uva/phts-risk-calculator/



### Deployment Verification

After deployment, verify all components are working:

1. **Lambda Function**: Check CloudWatch logs
2. **API Gateway**: Test endpoints (`/metadata`, `/risk`, `/causal`)
3. **S3 Dashboard**: Load HTML page and test risk calculation
4. **CORS**: Verify browser can call API without CORS errors

### Quick Deployment Script

For automated deployment, use the complete deployment script:

```bash
cd graft-loss/cohort_analysis/calculator/risk_dashboard
./deploy_complete.sh
```

This script automates all deployment steps.

### Documentation

For detailed deployment instructions, see:
- `docs/calculator/README_deployment.md` - Complete deployment guide
- `risk_dashboard/README_DEPLOYMENT.md` - Deployment reference
- `risk_dashboard/README_ARCHITECTURE.md` - Architecture overview

# Final Step: Shutdown EC2 Instance

In [ ]:
# Set SHUTDOWN_EC2 = True to enable, False to disable
SHUTDOWN_EC2 = True  # Change to True to enable auto-shutdown

print(f"\n{'=' * 80}")
print("Final Step: EC2 Instance Shutdown (Optional)")
print(f"{'=' * 80}")

if SHUTDOWN_EC2:
    print("\nShutting down EC2 instance...")
    print("-" * 80)
    
    import subprocess
    import shutil
    import os
    
    # Get instance ID from EC2 metadata service
    try:
        result = subprocess.run(
            ["curl", "-s", "http://169.254.169.254/latest/meta-data/instance-id"],
            capture_output=True,
            text=True,
            timeout=5
        )
        instance_id = result.stdout.strip()
        
        if instance_id and len(instance_id) > 0:
            print(f"Instance ID: {instance_id}")
            
            # Find AWS CLI
            aws_cmd = shutil.which("aws")
            if not aws_cmd:
                # Try common paths
                aws_paths = [
                    "/usr/local/bin/aws",
                    "/usr/bin/aws",
                    "/home/ec2-user/.local/bin/aws"
                ]
                for path in aws_paths:
                    if os.path.exists(path):
                        aws_cmd = path
                        break
            
            if aws_cmd:
                # Stop the instance (use terminate-instances for permanent deletion)
                shutdown_cmd = [aws_cmd, "ec2", "stop-instances", "--instance-ids", instance_id]
                
                print(f"Running: {' '.join(shutdown_cmd)}")
                result = subprocess.run(shutdown_cmd, capture_output=True, text=True)
                
                if result.returncode == 0:
                    print("\n✓ EC2 instance stop command sent successfully")
                    print("Instance will stop in a few moments.")
                    print("Note: This is a STOP (not terminate), so you can restart it later.")
                    logger.info(f"EC2 instance {instance_id} stop command sent successfully")
                else:
                    print(f"\n⚠ EC2 stop command returned exit code {result.returncode}.")
                    print("Check AWS credentials and permissions.")
                    if result.stderr:
                        print(f"Error: {result.stderr}")
                    logger.warning(f"EC2 stop command failed: {result.stderr}")
            else:
                print("\nWarning: AWS CLI not found. Cannot shutdown instance.")
                print("Install AWS CLI or ensure it's in your PATH.")
                logger.warning("AWS CLI not found, cannot shutdown EC2 instance")
        else:
            print("\nWarning: Could not determine instance ID. Skipping shutdown.")
            print("If you want to shutdown manually, use:")
            print("  aws ec2 stop-instances --instance-ids <your-instance-id>")
            logger.warning("Could not determine EC2 instance ID")
    except subprocess.TimeoutExpired:
        print("\nWarning: Timeout retrieving instance ID from metadata service.")
        print("If running on EC2, check that metadata service is accessible.")
        logger.warning("Timeout retrieving EC2 instance ID from metadata service")
    except Exception as e:
        print(f"\nWarning: Could not retrieve instance ID: {e}")
        print("If you want to shutdown manually, use:")
        print("  aws ec2 stop-instances --instance-ids <your-instance-id>")
        logger.warning(f"Error retrieving EC2 instance ID: {e}")
else:
    print("\nEC2 Auto-Shutdown: DISABLED")
    print("To enable auto-shutdown, set SHUTDOWN_EC2 = True in this cell.")
    print("Instance will continue running.")

print(f"\n{'=' * 80}")
print("Workflow Complete!")
print(f"{'=' * 80}")